In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 'http://minio:9000')
sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
sc._jsc.hadoopConfiguration().set('fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
sc._jsc.hadoopConfiguration().set('fs.s3a.connection.ssl.enabled', 'false')

print("spark session created")

order_id,customer_name,product,amount,order_date
ORD-0001,sarah johnson,Webcam,132.63,2024-11-17

In [ ]:
# READING  INTO DATAFRAME WITHOUT PROVIDING SCHEMA
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("s3a://glue-spark-etl-example-source/raw/orders/orders61.csv")

df.printSchema()

df.show()


In [ ]:
# READING  INTO DATAFRAME BY INERRING SCHEMA
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3a://glue-spark-etl-example-source/raw/orders/orders61.csv")

df.printSchema()

df.show()


In [ ]:
# CREATING DATAFRAME BY PROVIDING SCHEMA EXPLICITLY
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

schema = StructType([
    StructField("order_id",      StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("product",       StringType(), True),
    StructField("amount",        DoubleType(), True),
    StructField("order_date",    DateType(),   True)
])

df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("s3a://glue-spark-etl-example-source/raw/orders/orders61.csv")

df.printSchema()
df.show()


In [ ]:
# Read source CSV
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type='s3',
    connection_options={'paths': ['s3a://glue-spark-etl-example-source/raw/orders/orders61.csv']},
    format='csv',
    format_options={'withHeader': True, 'separator': ','}
)
dyf.printSchema()
dyf.show()

In [ ]:
dyf_methods = [m for m in dir(dyf) if not m.startswith('_')]
dyf_methods

## 1. applyMapping — rename, reorder and cast columns

In [ ]:
# (source_name, source_type, target_name, target_type)
mapped = dyf.apply_mapping([
    ('order_id',       'string', 'order_id',       'string'),
    ('customer_name',  'string', 'customer',       'string'),  # renamed
    ('product',        'string', 'product',        'string'),
    ('amount',         'string', 'amount',         'double'),  # cast to double
    ('order_date',     'string', 'order_date',     'date')     # cast to date
])
mapped.printSchema()
mapped.show()

## 2. resolveChoice — handle ambiguous or mixed-type columns

In [ ]:
# cast  — force a specific type
resolved_cast = dyf.resolveChoice(
    specs=[('amount', 'cast:double')]
)
resolved_cast.printSchema()
resolved_cast.show()
resolved_cast.toDF().show()

In [ ]:
# make_struct — keep all types as a struct (useful when column has mixed types)
resolved_struct = dyf.resolveChoice(
    specs=[('amount', 'make_struct')]
)
resolved_struct.printSchema()
resolved_struct.show()
resolved_struct.toDF().show()

In [ ]:

# project — pick one type and drop others
resolved_project = dyf.resolveChoice(
    specs=[('amount', 'project:double')]
)
resolved_project.printSchema()
resolved_project.show()
resolved_project.toDF().show()

## 3. drop_fields — drop unwanted columns

In [ ]:
dropped = mapped.drop_fields(['order_date'])
dropped.printSchema()
dropped.show()

## 4. select_fields — keep only specific columns

In [ ]:
selected = mapped.select_fields(['order_id', 'customer', 'amount'])
selected.printSchema()
selected.show(6)

## 5. filter — filter rows using a lambda

In [ ]:
high_value = mapped.filter(f=lambda row: row['amount'] > 400)
print(f'High value orders: {high_value.count()}')
high_value.show(3)

## 6. map — apply a transformation to each row

In [ ]:
def uppercase_customer(rec):
    rec['customer'] = rec['customer'].upper()
    return rec

uppercased = mapped.map(f=uppercase_customer)
uppercased.show(3)

## 7. rename_field — rename a single column

In [ ]:
renamed = mapped.rename_field('customer', 'customer_name')
renamed.printSchema()

## 8. Write transformed DynamicFrame back to MinIO

In [ ]:
glueContext.write_dynamic_frame.from_options(
    frame=mapped,
    connection_type='s3',
    connection_options={'path': 's3a://glue-spark-etl-example-target/silver/orders/'},
    format='parquet'
)
print('Write completed successfully')

In [68]:
spark.stop()